# 04 · Help BOBAI: seven ways (contest exercise)

**Style C: contest simulation**, a close copy of T24-O-BOBAI ("Help BOBAI: More classification", IOAI 2024
on-site): **cached embeddings**, a **frozen** 5-way `Linear` classifier, new classes with a handful of examples,
**no new learned parameters**, macro F1, and one prediction per line in a text file. The embeddings here come from
an image model rather than mBERT. **Traces to** `notes/ioai-task-patterns.md` P1, P2, P4, P6, P7.

## 1. Problem Description

Bob's client Amoira wants her image classifier extended from 5 to 7 classes **today**. The deployed service
cannot be redeployed with a new model, so the solution must be built around the classifier that is already there,
using only statistics of the data. For privacy, Bob only shares **precomputed embeddings** (ResNet18
pooled features) and never the images.

## 2. Dataset

| Array | Content |
|---|---|
| `E_old, y_old` | 2,500 embeddings `(512,)` for classes 0–4 (500 each), the data the head was trained on |
| `E_new, y_new` | **25** embeddings each for the new classes 5 and 6 |
| `base_clf` | frozen `nn.Linear(512, 5)` that expects **L2-normalised** embeddings |
| `E_test_a`, `E_test_b` | 1,750 embeddings each (classes 0–6, 250 per class), labels hidden |

Class names: 0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer, **5 dog, 6 frog**.

## 3. Task and rules (from T24-O-BOBAI)

- You may use `base_clf`, but you **cannot change its parameters or add any new learned parameters**. Nothing
  may be trained with gradients.
- You may compute averages, distances and other statistics of the embeddings (k-NN, prototypes, covariances,
  thresholds chosen on the training data).
- Inference on 500 embeddings must take under 2 minutes.

## 4. Submission

`submission.zip` containing `predictions_a.txt` and `predictions_b.txt`: one integer (0–6) per line, in the
order of the test arrays.

## 5. Scoring

Macro F1 over the 7 classes.
- Baseline (Leaderboard B): **0.49** (the official T24 baseline: with probability 2/7 guess a new class at random,
  otherwise trust the base classifier)
- Reference (Leaderboard B): **0.73**, which is your target

In [ ]:
import time, os, zipfile, random
T0 = time.time()
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

### Embeddings and the deployed classifier (DO NOT CHANGE)

Computing the embeddings takes about 30 s on a GPU. In the real task they were shipped precomputed.

In [ ]:
from datasets import load_dataset
from torchvision import models

def _setup():
    cifar = load_dataset("uoft-cs/cifar10")
    w = models.ResNet18_Weights.DEFAULT
    net = models.resnet18(weights=w); net.fc = nn.Identity(); net = net.to(device).eval(); pre = w.transforms()
    @torch.no_grad()
    def emb(ds):
        out = []
        for i in range(0, len(ds), 250):
            xb = torch.stack([pre(im.convert("RGB")) for im in ds[i:i + 250]["img"]]).to(device)
            out.append(net(xb).cpu())
        return torch.cat(out)
    rng = np.random.default_rng(2024)
    lab_tr, lab_te = np.array(cifar["train"]["label"]), np.array(cifar["test"]["label"])
    pick = lambda lab, c, k: rng.choice(np.flatnonzero(lab == c), k, replace=False)
    i_old = np.concatenate([pick(lab_tr, c, 500) for c in range(5)])
    i_new = np.concatenate([pick(lab_tr, c, 25) for c in (5, 6)])
    i_te = rng.permutation(np.concatenate([pick(lab_te, c, 500) for c in range(7)]))
    E_old, E_new, E_te = emb(cifar["train"].select(i_old)), emb(cifar["train"].select(i_new)), emb(cifar["test"].select(i_te))
    torch.manual_seed(2024)
    clf = nn.Linear(512, 5); opt = torch.optim.Adam(clf.parameters(), 1e-2)
    y_old = torch.tensor(lab_tr[i_old])
    for _ in range(400):
        loss = F.cross_entropy(clf(F.normalize(E_old, dim=1)), y_old); opt.zero_grad(); loss.backward(); opt.step()
    clf.requires_grad_(False)
    return E_old, y_old, E_new, torch.tensor(lab_tr[i_new]), E_te, lab_te[i_te], clf

E_old, y_old, E_new, y_new, _E_te, _y_te, base_clf = _setup()
E_test_a, E_test_b = _E_te[:1750], _E_te[1750:]
_ANSWERS = (_y_te[:1750], _y_te[1750:])          # the organisers' copy: do not use
del _E_te, _y_te
print(E_old.shape, E_new.shape, E_test_a.shape, E_test_b.shape)

### Baseline (replace this): the official T24 naive solution

In [ ]:
class SevenWayClassifier:
    def __init__(self, base_clf):
        self.base_clf = base_clf
    def __call__(self, E):                       # E: (n, 512) -> labels (n,)
        with torch.no_grad():
            base = self.base_clf(F.normalize(E, dim=1)).argmax(1).numpy()
        rnd = np.random.randint(0, 7, len(E))
        return np.where(rnd >= 5, rnd, base)

clf = SevenWayClassifier(base_clf)

### Local check: timing and a sanity score on the training embeddings

In [ ]:
t0 = time.time(); _ = clf(E_test_a[:500]); print(f"500 predictions in {time.time() - t0:.2f}s (limit 120s)")
p = clf(torch.cat([E_old, E_new])); t = torch.cat([y_old, y_new]).numpy()
print("macro F1 on training embeddings (optimistic):", round(f1_score(t, p, average="macro"), 3))
print("frozen head untouched:", not any(p.requires_grad for p in base_clf.parameters()))

### Submission

In [ ]:
pa, pb = clf(E_test_a), clf(E_test_b)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.writestr("predictions_a.txt", "\n".join(str(int(v)) for v in pa))
    z.writestr("predictions_b.txt", "\n".join(str(int(v)) for v in pb))
print("wrote submission.zip")

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    out = []
    with zipfile.ZipFile(zip_path) as z:
        for name, truth in zip(["predictions_a.txt", "predictions_b.txt"], _ANSWERS):
            try:
                p = np.array([int(v) for v in z.read(name).decode().split()])
                assert len(p) == len(truth)
                out.append(f1_score(truth, p, average="macro"))
            except Exception as e:
                print(name, "rejected:", e); out.append(0.0)
    return out

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
print(f"total runtime {(time.time() - T0) / 60:.1f} min")